In [ ]:
import sys
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')
sys.path.insert(0, '../../../../..')

from load_data.load_conformal_data import load_conformance_results
from src.conformance_analysis.risk_model import DataFrameConstruction, ConformalAnalysisVisualizations, LogisticRegressionModel
from src.conformance_analysis.deviation_model import DeviationPredictionCalibration

# Load the calibration dataset

In [ ]:
# Load the validation fitness score results:

# Helpdesk
# path_fitness_results = "../../../../../data/Helpdesk/proact_conf_check_v2/offline_analysis/"
# New path: store model
# path_model = "./results/Helpdesk/crc_logistic_model.pkl"
# New path: store calibration results
# path_dev_threshs = "./results/Helpdesk/deviation_labels_thresholds.json"

# Sepsis
# path_fitness_results = "../../../../../data/Sepsis/proact_conf_check_v2/offline_analysis"
# New path: store model
# path_model = "./results/Sepsis/crc_logistic_model.pkl"
# New path: store calibration results
# path_dev_threshs = "./results/Sepsis/deviation_labels_thresholds.json"

# BPIC20:
# path_fitness_results = "../../../../../data/BPIC20/proact_conf_check_v2/offline_analysis"
# New path: store model
# path_model = "./results/BPIC20/crc_logistic_model.pkl"
# New path: store calibration results
# path_dev_threshs = "./results/BPIC20/deviation_labels_thresholds.json"

# Repair:
path_fitness_results = "../../../../../data/repair_shop/proact_conf_check_v2/offline_analysis"
# New path: store model
path_model = "./results/Repair/crc_logistic_model.pkl"
# New path: store calibration results
path_dev_threshs = "./results/Repair/deviation_labels_thresholds.json"


In [ ]:
# All results of Prob. Suffix Pred. + Alignment-Based Conformance Checking:
# uses only the D_risk set
risk_data_results = load_conformance_results(path=path_fitness_results)

# Case ids
res_case_id = risk_data_results['case_id']
# Target conformance results
res_target_conf = risk_data_results['target_conformance']
res_target_conf_suffix_fit = [res['suffix_fitness'] for res in res_target_conf]
# Most-likely
res_ml_conf = risk_data_results['ml_conformance']
res_ml_conf_suffix_fit = [res['suffix_fitness'] for res in res_ml_conf]
# Samples
res_smpl_conf = risk_data_results['samples_conformance']
res_smpl_conf_suffix_fit = [[r['suffix_fitness'] for r in res] for res in res_smpl_conf]

In [ ]:
# Design choices:
# Set q-level (percentage of suffix samples to check with alpha risk lower fitness score) of risk safe and determine r_q (fitness value):
# e.g.: the 95% percent worst
alpha_risk = 1.0

# Set alpha for logistic regression calibration: for example with prob: 1-alpha, I ensure that my prd lies in the right set.
# alpha_expected_loss = 0.05
# alpha_expected_loss = 0.15
alpha_expected_loss = 0.25


# Build dataframe for risk-controlled conformance model

In [ ]:
# Create dataframe for training and calibration datasets:
dc = DataFrameConstruction(conformance_results=risk_data_results)

# Emprical Alpha qunatiles for (suffix) fitness score thresholds:
alpha_risk = alpha_risk

# change aggregation to 'mean'
aggregation_samples = 'median'

# Compute based on alpha the suffix fitness values:
emp_results = dc.empirical_quantile_thresholds(alpha_risk=alpha_risk, 
                                               aggregation=aggregation_samples)
print("Empirical risk value results: ", emp_results)

In [ ]:
# Suffix fitness score distribution
cav = ConformalAnalysisVisualizations(sampled_fitness=res_smpl_conf_suffix_fit,
                                      target_fitness=res_target_conf_suffix_fit,
                                      ml_fitness=res_ml_conf_suffix_fit)
  
# Empirical distribution of fitness scores and quantile thresholds:
# ! Only the target fitness per case is used to determine if a case is risk or safe
q_risk = cav.plot_distribution(alpha_risk=alpha_risk,
                               # Print median aggregated sampled fitness scores distribution:
                               aggregation=aggregation_samples)

In [ ]:
# Create dataframe for training and calibration datasets:
dc = DataFrameConstruction(conformance_results=risk_data_results)

# Add as threshold the determined q_risk
df = dc.samples_to_dataframe(q_risk=q_risk,
                             target_col='y_safe_case',
                             include_tail_features=True)
# All values that have fitness >= risk threshold are save!
print(df.head(5))

# Suppose df is your dataframe and 'y' is the target column
X = df.drop(columns='y_safe_case')
# 1 = safe, 0 = risk
y = df['y_safe_case']

# Split the conformance calibration dataframe into train (train log reg on that) and conformal calibration (calibrate log reg) datsets:
X_train, X_cal, y_train, y_cal = train_test_split(X,
                                                  y, 
                                                  test_size=0.2,      # 20% of all data in total.
                                                  random_state=42,    # ensures reproducibility,
                                                  stratify=y          # optional: keeps class balance if classification
                                                  )
print("Shape of train and calibration sets:", X_train.shape, X_cal.shape)

## Train risk classifier model

In [ ]:
# Logistic regression model training:
df_train = X_train.copy()
df_train['y_safe_case'] = y_train

lm = LogisticRegressionModel(alpha_quantile_risk=alpha_risk, risk_fitness_threshold=q_risk)
lm.fit_from_dataframe(df_train, target_col='y_safe_case', calibrate=False)
print('model intercept :', lm.classifier.intercept_)
print('model coefficients : ', lm.classifier.coef_)

In [ ]:
# Prediction accuracy on conformal calibration subset of full calibration set:
# Calibration dataframe
df_cal = X_cal.copy()
df_cal['y_safe_case'] = y_cal

cal_score = lm.pipeline.score(X_cal, y_cal)
print("Cal accuracy:", cal_score)

# Preditcition probabilities on train and cal sets:
y_prob_cal = lm.pipeline.predict_proba(X_cal)[:,1]

plt.hist(y_prob_cal, bins=20, alpha=0.5, label='Cal')
plt.xlabel("Predicted probability")
plt.ylabel("Count")
plt.legend()
plt.show()

## Calibrate the model using conformal risk control and save

In [ ]:
# Conformal prediction calibration:
cal_res = lm.calibrate_conformal_threshold(X_cal=df_cal.drop(columns='y_safe_case'),
                                           y_cal=df_cal['y_safe_case'],
                                           alpha=alpha_expected_loss)

# Use CRC to control false-safe rate among predicted-safe cases
# cal_res = lm.calibrate_crc_safe_threshold(X_cal=df_cal.drop(columns='y_safe_case'),
#                                           y_cal=df_cal['y_safe_case'],
#                                           alpha=alpha_expected_loss)

print(f"All cases predicted with p >= {lm.calibration_info['threshold']} are classified as safe.")

# Save the model
lm.save(path_model)

## Calibrate binary prediction on deviation labels

In [ ]:
# Calibrate the deviation predictions for all labels using the F_beta score:
# Create dataframe for risk examples only:
dc = DataFrameConstruction(conformance_results=risk_data_results) 

# Add as threshold the determined q_risk fitness score that is stored in the logistic regression model, to generate the targets
# lm.calibration_info['threshold'] is a *probability* threshold (for predicting safe), not a fitness threshold.
df = dc.samples_to_dataframe(q_risk=lm.risk_fitness_threshold,
                             target_col='y_safe_case',
                             include_tail_features=True)

# Get all cases that are risk and count therefore for deviation prediction:
risks = {'case_id': [],
         'target_conformance': [],
         'ml_conformance': [],
         'samples_conformance': []} 

labels, probs = lm.predict_with_threshold(X=df)

for i in range(len(labels)):
    if labels[i] == 0:
        risks['case_id'].append(res_case_id[i])
        risks['target_conformance'].append(res_target_conf[i])
        risks['ml_conformance'].append(res_ml_conf[i])
        risks['samples_conformance'].append(res_smpl_conf[i])

In [ ]:
# Compute deviation label thresholds        
dpc = DeviationPredictionCalibration(risk_conformance_results=risks)

deviation_thresholds = dpc.find_optimal_thresholds(preference='balanced')
print("Calibrated deviation label thresholds for deviation prediction:", deviation_thresholds)
print("if p(case) -> risk -> p(dev) >= t -> predicted as TRUE")

dpc.save(path=path_dev_threshs, thresholds=deviation_thresholds)